# 🧠 Notebook 1 — Download & Pré-processamento

Parte 1 de 4 do pipeline de predição de crises.

**Datasets:** CHB-MIT, Siena, **Mendeley (AUB)** e SeizeIT2.

Este notebook: baixa só os arquivos com crise, lê as anotações, **reamostra tudo para um fs comum**, filtra (notch + passa-banda) e salva o **Nível 1 (L1)** — sinal filtrado contínuo + labels amostra-a-amostra + nomes dos canais. Os notebooks seguintes leem o L1 do disco.

## 1.1 Imports

In [1]:
%pip install numpy pandas matplotlib seaborn mne PyWavelets boto3 scipy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, io, warnings, gc, html
import html.parser
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import pywt
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch, resample_poly
from scipy.stats  import kurtosis as sp_kurtosis, skew as sp_skew
from tqdm.auto    import tqdm

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✅ Imports OK")

✅ Imports OK


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2 Configuração global

Usada por todos os 4 notebooks (cada um é independente e redefine a config no topo).

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Diretórios ───────────────────────────────────
ROOT_DIR    = 'data'
CHBMIT_DIR  = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR   = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR= os.path.join(ROOT_DIR, 'mendeley')
L1_DIR      = os.path.join(ROOT_DIR, 'level1_signals')
L2_DIR      = os.path.join(ROOT_DIR, 'level2_windows')
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')   # subpastas por nível: features/<level>/
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR,
          L1_DIR, L2_DIR, FEAT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Filtragem ──────────────────────────────────
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}

# Reamostragem comum (Siena=512, Mendeley=500, demais=256) → fs único para
# tornar features espectrais comparáveis e permitir juntar datasets.
TARGET_FS = 256

# ── Janelamento ────────────────────────────────
WIN_SEC, OVERLAP = 4, 0.50

# ── Rotulagem ──────────────────────────────────
PRE_SEC, POST_SEC, IGAP_SEC = 10*60, 10*60, 10*60
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)
LBL_NAMES = {v: k for k, v in LBL.items()}

# ── Cap de interictal na extração (controla custo do SeizeIT2) ──────────
MAX_INTER_RATIO = 10   # máx. interictal:eventos por paciente (None desliga)

# ── PACIENTES POR DATASET ──────────────────────────────
PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14'],
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005'],
}
PILOT = {k: v[0] for k, v in PATIENTS.items()}
SEIZEIT_SESSION = 'ses-01'

# ── NÍVEIS DE REDUÇÃO DE CANAIS = subconjuntos de REGIÕES cerebrais ────────
# Por que regiões e não eletrodos fixos? Os datasets usam montagens diferentes
# (CHB-MIT é bipolar, Siena/Mendeley referencial, SeizeIT2 behind-the-ear), então
# "o mesmo eletrodo" não existe entre todos. Mapear cada canal → região e agregar
# por região (1) funciona em qualquer montagem, (2) dá vetor de tamanho fixo para
# poder juntar datasets, e (3) mantém a interpretabilidade (SHAP por região).
REGIONS = ['frontal', 'temporal', 'central', 'parietal', 'occipital']

LEVELS = {
    'R5': ['frontal', 'temporal', 'central', 'parietal', 'occipital'],  # completo (~19 ch)
    'R3': ['frontal', 'temporal', 'central'],                           # reduzido (~8 ch)
    'R2': ['frontal', 'temporal'],                                      # vestível (~4 ch)
    'R1': ['temporal'],                                                 # ultra-compacto / SeizeIT2
}
# Quais datasets participam de cada nível. SeizeIT2 (behind-the-ear, ~região
# temporal) só entra no nível mínimo R1, que é onde ele faz sentido físico.
LEVEL_DATASETS = {
    'R5': ['CHBMIT', 'Siena', 'Mendeley'],
    'R3': ['CHBMIT', 'Siena', 'Mendeley'],
    'R2': ['CHBMIT', 'Siena', 'Mendeley'],
    'R1': ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2'],
}

# ── Mapa eletrodo → região (sistema 10-20; aceita variantes T7/T3 etc.) ────
ELECTRODE_REGION = {}
for _e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']:           ELECTRODE_REGION[_e]='frontal'
for _e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[_e]='temporal'
for _e in ['c3','cz','c4','fc1','fc2','cp1','cp2']:                     ELECTRODE_REGION[_e]='central'
for _e in ['p3','pz','p4','po3','po4']:                                 ELECTRODE_REGION[_e]='parietal'
for _e in ['o1','o2','oz']:                                             ELECTRODE_REGION[_e]='occipital'

print("✅ Configurações OK")
print(f"   fs alvo     : {TARGET_FS} Hz | Janela {WIN_SEC}s overlap {int(OVERLAP*100)}%")
print(f"   Pré/Pós     : {PRE_SEC//60}/{POST_SEC//60} min | cap interictal {MAX_INTER_RATIO}:1")
print(f"   Níveis      : " + " | ".join(f"{k}({len(v)} reg)" for k,v in LEVELS.items()))
for k, v in PATIENTS.items():
    print(f"   {k:9s}: {v}")

✅ Configurações OK
   fs alvo     : 256 Hz | Janela 4s overlap 50%
   Pré/Pós     : 10/10 min | cap interictal 10:1
   Níveis      : R5(5 reg) | R3(3 reg) | R2(2 reg) | R1(1 reg)
   CHBMIT   : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05']
   Siena    : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06']
   Mendeley : ['p10', 'p11', 'p12', 'p13', 'p14']
   SeizeIT2 : ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005']


In [4]:
def normalize_electrode(name):
    """Limpa um nome de canal: 'EEG Fp1-REF' -> 'fp1'. Para bipolar 'F7-T3'
    devolve o PRIMEIRO nó ('f7'), que define a região do canal."""
    s = str(name).lower()
    s = s.replace('eeg', ' ').replace('-ref', ' ').replace('-le', ' ').replace('ref', ' ')
    s = s.strip()
    # bipolar 'a-b' -> primeiro nó
    if '-' in s:
        s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def channel_region(name, dataset=None):
    """Região cerebral de um canal. SeizeIT2 (behind-the-ear) -> temporal."""
    if dataset == 'SeizeIT2':
        return 'temporal'
    return ELECTRODE_REGION.get(normalize_electrode(name), None)

def regions_present(ch_names, dataset=None):
    """Dict região -> lista de índices de canais naquela região."""
    out = {r: [] for r in REGIONS}
    for i, nm in enumerate(ch_names):
        r = channel_region(nm, dataset)
        if r in out:
            out[r].append(i)
    return out

## 2 — Download dos datasets (só arquivos com crise)

### 2.1 CHB-MIT (S3 PhysioNet)

In [5]:
def s3_client():
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED))

def download_chbmit_patient(patient, local_root=CHBMIT_DIR):
    """
    Baixa, para UM paciente CHB-MIT, apenas os EDFs que contêm crise.
    Usa o índice oficial RECORDS-WITH-SEIZURES para filtrar — não baixa
    arquivos sem crise.
    """
    s3      = s3_client()
    bucket  = 'physionet-open'
    prefix  = 'chbmit/1.0.0/'
    pat_dir = os.path.join(local_root, patient)
    os.makedirs(pat_dir, exist_ok=True)

    # RECORDS-WITH-SEIZURES — lista global de gravações COM crise
    rec_local = os.path.join(local_root, 'RECORDS-WITH-SEIZURES')
    if not os.path.exists(rec_local):
        print("⬇️  Baixando RECORDS-WITH-SEIZURES ...")
        s3.download_file(bucket, prefix + 'RECORDS-WITH-SEIZURES', rec_local)

    with open(rec_local) as f:
        seizure_recs = [l.strip() for l in f if l.strip().startswith(patient + '/')]

    # summary.txt (tem os tempos de início/fim das crises)
    sum_key   = f'{prefix}{patient}/{patient}-summary.txt'
    sum_local = os.path.join(pat_dir, f'{patient}-summary.txt')
    if not os.path.exists(sum_local):
        s3.download_file(bucket, sum_key, sum_local)

    # Baixa SOMENTE os EDFs com crise
    downloaded = 0
    for rec in tqdm(seizure_recs, desc=f'CHB-MIT {patient}', leave=False):
        fname = rec.split('/')[-1]
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try:
                s3.download_file(bucket, prefix + rec, local)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {rec}: {e}")
    print(f"✅ CHB-MIT {patient} — {len(seizure_recs)} EDFs com crise | {downloaded} novos")
    return pat_dir

# ── Baixa todos os pacientes CHB-MIT da lista ────────────────────────────────
for _p in PATIENTS['CHBMIT']:
    download_chbmit_patient(_p)

✅ CHB-MIT chb01 — 7 EDFs com crise | 0 novos


✅ CHB-MIT chb02 — 3 EDFs com crise | 0 novos


✅ CHB-MIT chb03 — 7 EDFs com crise | 0 novos


✅ CHB-MIT chb04 — 3 EDFs com crise | 0 novos


✅ CHB-MIT chb05 — 5 EDFs com crise | 0 novos


### 2.2 Siena (HTTPS PhysioNet)

In [11]:
SIENA_BASE = 'https://physionet.org/files/siena-scalp-eeg/1.0.0/'

class _LinkParser(html.parser.HTMLParser):
    def __init__(self):
        super().__init__(); self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            for attr, val in attrs:
                if attr == 'href' and not val.startswith(('?','/','..')):
                    self.links.append(val)

def _siena_seizure_edfs(patient, seizures_list_text):
    """Extrai do Seizures-list-PNxx.txt os nomes de EDF que têm crise."""
    edfs = set()
    for line in seizures_list_text.splitlines():
        m = re.search(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
        if m:
            edfs.add(m.group(1).strip())
    return edfs

def download_siena_patient(patient, local_root=SIENA_DIR):
    """
    Baixa, para UM paciente Siena: o Seizures-list (anotações) e APENAS os
    EDFs que aparecem nele (ou seja, com crise). Evita baixar gravações sem crise.
    """
    pat_dir  = os.path.join(local_root, patient)
    os.makedirs(pat_dir, exist_ok=True)
    base_url = SIENA_BASE + patient + '/'

    try:
        with urllib.request.urlopen(base_url, timeout=30) as r:
            content = r.read().decode('utf-8')
    except Exception as e:
        print(f"⚠️  Não foi possível listar {base_url}: {e}")
        return pat_dir

    p = _LinkParser(); p.feed(content)
    all_files = p.links

    # 1) baixa primeiro os .txt (anotações) para saber quais EDFs têm crise
    txt_files = [l for l in all_files if l.endswith('.txt')]
    for fname in txt_files:
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try: urllib.request.urlretrieve(base_url + fname, local)
            except Exception as e: print(f"  ⚠️  {fname}: {e}")

    # 2) descobre EDFs com crise a partir do Seizures-list-PNxx.txt
    seizure_edfs = set()
    sl_path = os.path.join(pat_dir, f'Seizures-list-{patient}.txt')
    if os.path.exists(sl_path):
        with open(sl_path, encoding='utf-8', errors='ignore') as f:
            seizure_edfs = _siena_seizure_edfs(patient, f.read())

    # fallback: se não achou a lista, baixa todos os EDFs (comportamento antigo)
    edf_files = [l for l in all_files if l.endswith('.edf')]
    target_edfs = [e for e in edf_files if (not seizure_edfs) or (e in seizure_edfs)]

    downloaded = 0
    for fname in tqdm(target_edfs, desc=f'Siena {patient}', leave=False):
        local = os.path.join(pat_dir, fname)
        if not os.path.exists(local):
            try:
                urllib.request.urlretrieve(base_url + fname, local)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {fname}: {e}")
    print(f"✅ Siena {patient} — {len(target_edfs)} EDFs com crise | {downloaded} novos")
    return pat_dir

for _p in PATIENTS['Siena']:
    download_siena_patient(_p)

✅ Siena PN00 — 5 EDFs com crise | 0 novos


✅ Siena PN01 — 0 EDFs com crise | 0 novos


✅ Siena PN03 — 2 EDFs com crise | 0 novos


✅ Siena PN05 — 3 EDFs com crise | 0 novos


KeyboardInterrupt: 

### 2.3 SeizeIT2 (S3 OpenNeuro)

In [7]:
SEIZEIT_BUCKET  = 'openneuro.org'
SEIZEIT_DATASET = 'ds005873'
SEIZEIT_SESSION = 'ses-01'

# Tipos de evento que SÃO crise no SeizeIT2 (BIDS). Tudo o que não estiver aqui
# (ex.: 'impd' = impedance check, 'bckg' = background) é IGNORADO.
SEIZEIT_SZ_TYPES = {'sz', 'sz_foc_a', 'sz_foc_ia', 'sz_gen', 'szt1', 'szt2'}

def _tsv_has_seizure(tsv_bytes):
    """Lê um _events.tsv (em bytes) e diz se contém ao menos uma crise válida."""
    import io
    try:
        df = pd.read_csv(io.BytesIO(tsv_bytes), sep='\t')
    except Exception:
        return False
    if 'eventType' not in df.columns:
        return False
    et = df['eventType'].astype(str).str.lower().str.strip()
    # crise = começa com 'sz' E não é um tipo claramente não-crise
    is_sz = et.str.startswith('sz')
    return bool(is_sz.any())

def download_seizeit2_subject(subject, local_root=SEIZEIT_DIR):
    """
    Baixa, para UM sujeito SeizeIT2: primeiro os _events.tsv (leves), decide
    quais runs têm crise, e então baixa SOMENTE os _eeg.edf (+ .json) desses runs.
    Evita baixar EDFs grandes de gravações sem crise.
    """
    s3      = s3_client()
    prefix  = f'{SEIZEIT_DATASET}/{subject}/{SEIZEIT_SESSION}/eeg/'
    sub_dir = os.path.join(local_root, subject, SEIZEIT_SESSION, 'eeg')
    os.makedirs(sub_dir, exist_ok=True)

    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=SEIZEIT_BUCKET, Prefix=prefix):
        keys.extend(o['Key'] for o in page.get('Contents', []))

    # 1) baixa todos os _events.tsv (são pequenos)
    tsv_keys = [k for k in keys if k.endswith('_events.tsv')]
    seizure_runs = set()   # prefixos de run com crise, ex 'sub-001_..._run-03'
    for key in tsv_keys:
        fname = key.split('/')[-1]
        local = os.path.join(sub_dir, fname)
        if not os.path.exists(local):
            try: s3.download_file(SEIZEIT_BUCKET, key, local)
            except Exception as e: print(f"  ⚠️  {fname}: {e}")
        # checa se tem crise
        try:
            with open(local, 'rb') as f:
                if _tsv_has_seizure(f.read()):
                    seizure_runs.add(fname.replace('_events.tsv', ''))
        except Exception:
            pass

    # 2) baixa apenas os _eeg.edf (+ sidecar .json) dos runs com crise
    downloaded = 0
    for key in tqdm(keys, desc=f'SeizeIT2 {subject}', leave=False):
        fname = key.split('/')[-1]
        run_prefix = fname.replace('_eeg.edf','').replace('_eeg.json','')
        is_eeg = fname.endswith('_eeg.edf') or fname.endswith('_eeg.json')
        if is_eeg and run_prefix not in seizure_runs:
            continue   # pula gravações sem crise
        local = os.path.join(sub_dir, fname)
        if not os.path.exists(local):
            try:
                s3.download_file(SEIZEIT_BUCKET, key, local)
                downloaded += 1
            except Exception as e:
                print(f"  ⚠️  {fname}: {e}")
    print(f"✅ SeizeIT2 {subject} — {len(seizure_runs)} runs com crise | {downloaded} novos arquivos")
    return sub_dir

for _p in PATIENTS['SeizeIT2']:
    download_seizeit2_subject(_p)

✅ SeizeIT2 sub-001 — 4 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-002 — 7 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-003 — 1 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-004 — 1 runs com crise | 0 novos arquivos


✅ SeizeIT2 sub-005 — 2 runs com crise | 0 novos arquivos


### 2.4 Mendeley — Epileptic EEG Dataset (AUB)

Baixado via API pública do Mendeley (cloudscraper contorna o Cloudflare). As anotações vêm numa planilha `Seizures_Information.xlsx` (tempos em fração de dia).

In [8]:
# ── Download Mendeley (Epileptic EEG Dataset, AUB) via API pública ─────────
# Usa cloudscraper para contornar o Cloudflare do data.mendeley.com.
try:
    import cloudscraper
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'cloudscraper', '--quiet'])
    import cloudscraper

MEND_DATASET_ID = '5pc2j46cbc'
MEND_VERSION    = 1
MEND_DOC_FOLDER = '465c0896-7d08-4fbe-8ee3-378beca656d5'
MEND_EDF_FOLDER = '3bc6cc31-0156-490b-8e39-4b740598b289'
_MEND_HEADERS   = {"Accept": "application/vnd.mendeley-public-dataset.1+json"}

def _mendeley_list(folder_id):
    scraper = cloudscraper.create_scraper()
    url = (f"https://data.mendeley.com/public-api/datasets/{MEND_DATASET_ID}"
           f"/files?folder_id={folder_id}&version={MEND_VERSION}")
    r = scraper.get(url, headers=_MEND_HEADERS); r.raise_for_status()
    return r.json()

def download_mendeley(patients=None, local_root=MENDELEY_DIR):
    """Baixa o Seizures_Information.xlsx e os EDFs dos pacientes pedidos.
    patients: lista de prefixos como ['p10','p11']; None = todos os EDFs.
    Cada paciente tem vários *_RecordN.edf; baixamos todos os records do prefixo."""
    scraper = cloudscraper.create_scraper()
    doc_dir = os.path.join(local_root, 'Documentation')
    edf_dir = os.path.join(local_root, 'Raw_EDF_Files')
    os.makedirs(doc_dir, exist_ok=True); os.makedirs(edf_dir, exist_ok=True)

    # 1) planilha de anotações
    for fobj in _mendeley_list(MEND_DOC_FOLDER):
        if fobj['filename'] == 'Seizures_Information.xlsx':
            dst = os.path.join(doc_dir, 'Seizures_Information.xlsx')
            if not os.path.exists(dst):
                print("⬇️  Seizures_Information.xlsx ...")
                with open(dst, 'wb') as f:
                    f.write(scraper.get(fobj['content_details']['download_url']).content)

    # 2) EDFs (filtra pelos prefixos de paciente)
    edf_files = _mendeley_list(MEND_EDF_FOLDER)
    def _keep(fn):
        if patients is None:
            return True
        return any(fn.lower().startswith(p.lower() + '_') or
                   fn.lower().startswith(p.lower() + 'record') for p in patients)
    targets = [f for f in edf_files if f['filename'].lower().endswith('.edf') and _keep(f['filename'])]
    dl = 0
    for fobj in tqdm(targets, desc='Mendeley', leave=False):
        dst = os.path.join(edf_dir, fobj['filename'])
        if not os.path.exists(dst):
            with open(dst, 'wb') as f:
                f.write(scraper.get(fobj['content_details']['download_url']).content)
            dl += 1
    print(f"✅ Mendeley — {len(targets)} EDFs alvo | {dl} novos")

download_mendeley(PATIENTS['Mendeley'])

✅ Mendeley — 16 EDFs alvo | 15 novos


## 3 — Parsers de anotação (intervalos de crise por arquivo)

### 3.1 CHB-MIT

In [9]:
def parse_chbmit_summary(patient_dir, patient):
    """Retorna { 'chb01_03.edf': [(start_s, end_s), ...] } — só arquivos com crise."""
    path = os.path.join(patient_dir, f'{patient}-summary.txt')
    seizures = {}
    cur_file, n_seiz = None, 0
    with open(path, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if line.startswith('File Name:'):
                cur_file = line.split(':', 1)[1].strip(); n_seiz = 0
            elif line.startswith('Number of Seizures in File:'):
                n_seiz = int(line.split(':', 1)[1].strip())
                if n_seiz > 0: seizures.setdefault(cur_file, [])
            elif cur_file and n_seiz > 0:
                if 'Seizure' in line and 'Start Time' in line:
                    t = int(re.search(r'(\d+)', line.split(':',1)[1]).group(1))
                    seizures[cur_file].append([t, None])
                elif 'Seizure' in line and 'End Time' in line:
                    t = int(re.search(r'(\d+)', line.split(':',1)[1]).group(1))
                    if seizures[cur_file] and seizures[cur_file][-1][1] is None:
                        seizures[cur_file][-1][1] = t
    return {f: [(s,e) for s,e in lst if e is not None]
            for f, lst in seizures.items()
            if any(e is not None for s,e in lst)}

# ── Anotações de TODOS os pacientes CHB-MIT ──────────────────────────────────
# chbmit_ann_all = { 'chb01': { 'chb01_03.edf': [(s,e),...], ... }, ... }
chbmit_ann_all = {}
for _p in PATIENTS['CHBMIT']:
    ann = parse_chbmit_summary(os.path.join(CHBMIT_DIR, _p), _p)
    chbmit_ann_all[_p] = ann
    print(f"✅ CHB-MIT {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {fname}  →  {s}s – {e}s  ({e-s}s)")

✅ CHB-MIT chb01 — 7 arquivos com crise
   chb01_03.edf  →  2996s – 3036s  (40s)
   chb01_04.edf  →  1467s – 1494s  (27s)
   chb01_15.edf  →  1732s – 1772s  (40s)
   chb01_16.edf  →  1015s – 1066s  (51s)
   chb01_18.edf  →  1720s – 1810s  (90s)
   chb01_21.edf  →  327s – 420s  (93s)
   chb01_26.edf  →  1862s – 1963s  (101s)
✅ CHB-MIT chb02 — 3 arquivos com crise
   chb02_16.edf  →  130s – 212s  (82s)
   chb02_16+.edf  →  2972s – 3053s  (81s)
   chb02_19.edf  →  3369s – 3378s  (9s)
✅ CHB-MIT chb03 — 7 arquivos com crise
   chb03_01.edf  →  362s – 414s  (52s)
   chb03_02.edf  →  731s – 796s  (65s)
   chb03_03.edf  →  432s – 501s  (69s)
   chb03_04.edf  →  2162s – 2214s  (52s)
   chb03_34.edf  →  1982s – 2029s  (47s)
   chb03_35.edf  →  2592s – 2656s  (64s)
   chb03_36.edf  →  1725s – 1778s  (53s)
✅ CHB-MIT chb04 — 3 arquivos com crise
   chb04_05.edf  →  7804s – 7853s  (49s)
   chb04_08.edf  →  6446s – 6557s  (111s)
   chb04_28.edf  →  1679s – 1781s  (102s)
   chb04_28.edf  →  3782s – 389

### 3.2 Siena

In [10]:
def _hms_to_sec(s):
    """Converte 'HH.MM.SS' ou 'HH:MM:SS' para segundos inteiros."""
    s = s.strip().replace(':', '.')
    parts = s.split('.')
    if len(parts) == 3:
        h, m, sec = int(parts[0]), int(parts[1]), int(parts[2])
        return h * 3600 + m * 60 + sec
    return 0

def _clock_offset(reg_start_s, event_s):
    """
    Calcula offset em segundos de event_s em relação a reg_start_s,
    tratando cruzamento de meia-noite (event_s < reg_start_s).
    """
    diff = event_s - reg_start_s
    if diff < 0:          # passou da meia-noite
        diff += 24 * 3600
    return diff

def parse_siena_seizure_list(patient_dir, patient):
    """
    Lê Seizures-list-PNXX.txt e retorna:
        { 'PN00-1.edf': [(start_rel_s, end_rel_s), ...], ... }
    Os tempos são relativos ao início da gravação de cada arquivo.
    """
    fname_txt = f'Seizures-list-{patient}.txt'
    path = os.path.join(patient_dir, fname_txt)
    if not os.path.exists(path):
        print(f"⚠️  {fname_txt} não encontrado em {patient_dir}")
        return {}

    seizures   = {}
    cur_file   = None
    reg_start  = None

    with open(path, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            m = re.match(r'File name:\s*(\S+\.edf)', line, re.IGNORECASE)
            if m:
                cur_file  = m.group(1).strip()
                reg_start = None
                seizures.setdefault(cur_file, [])
                continue

            m = re.match(r'Registration start time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m:
                reg_start = _hms_to_sec(m.group(1))
                continue

            m = re.match(r'Seizure start time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur_file is not None and reg_start is not None:
                t = _clock_offset(reg_start, _hms_to_sec(m.group(1)))
                seizures[cur_file].append([t, None])
                continue

            m = re.match(r'Seizure end time:\s*([\d.:]+)', line, re.IGNORECASE)
            if m and cur_file is not None and reg_start is not None:
                t = _clock_offset(reg_start, _hms_to_sec(m.group(1)))
                if seizures[cur_file] and seizures[cur_file][-1][1] is None:
                    seizures[cur_file][-1][1] = t

    return {f: [(s,e) for s,e in lst if e is not None and e > s]
            for f, lst in seizures.items()
            if any(e is not None and e > s for s,e in lst)}

# ── Anotações de TODOS os pacientes Siena ────────────────────────────────────
siena_ann_all = {}
for _p in PATIENTS['Siena']:
    ann = parse_siena_seizure_list(os.path.join(SIENA_DIR, _p), _p)
    siena_ann_all[_p] = ann
    print(f"✅ Siena {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {fname}  →  {s}s – {e}s  ({e-s}s)")

✅ Siena PN00 — 5 arquivos com crise
   PN00-1.edf  →  1143s – 1213s  (70s)
   PN00-2.edf  →  1220s – 1274s  (54s)
   PN00-3.edf  →  765s – 4425s  (3660s)
   PN00-4.edf  →  1006s – 1080s  (74s)
   PN00-5.edf  →  904s – 971s  (67s)
✅ Siena PN01 — 0 arquivos com crise
✅ Siena PN03 — 2 arquivos com crise
   PN03-1.edf  →  38673s – 38784s  (111s)
   PN03-2.edf  →  34921s – 35054s  (133s)
✅ Siena PN05 — 3 arquivos com crise
   PN05-2.edf  →  7163s – 7198s  (35s)
   PN05-3.edf  →  6836s – 6866s  (30s)
   PN05-4.edf  →  3608s – 3647s  (39s)
✅ Siena PN06 — 5 arquivos com crise
   PNO6-1.edf  →  5583s – 5647s  (64s)
   PNO6-2.edf  →  8860s – 8929s  (69s)
   PN06-3.edf  →  6275s – 6317s  (42s)
   PNO6-4.edf  →  5939s – 6002s  (63s)
   PN06-5.edf  →  4783s – 4827s  (44s)


### 3.3 SeizeIT2

In [ ]:
def parse_seizeit2_subject(subject_eeg_dir):
    """
    Varre os *_events.tsv do sujeito e retorna { edf_name: [(onset, onset+dur), ...] }.

    Blindagem contra eventos NÃO-crise: só consideramos linhas cujo eventType
    começa com 'sz' (crise no padrão BIDS do SeizeIT2). Eventos como 'impd'
    (impedance check), 'bckg' (background), movimento etc. são DESCARTADOS,
    pois não começam com 'sz'.
    """
    seizures = {}
    tsv_files = sorted(f for f in os.listdir(subject_eeg_dir)
                       if f.endswith('_events.tsv'))
    for tsv in tsv_files:
        edf_name = tsv.replace('_events.tsv', '_eeg.edf')
        try:
            df = pd.read_csv(os.path.join(subject_eeg_dir, tsv), sep='\t')
        except Exception:
            continue
        if 'eventType' not in df.columns:
            continue
        et = df['eventType'].astype(str).str.lower().str.strip()
        # SÓ crises: eventType começa com 'sz' (descarta impd, bckg, etc.)
        sz = df[et.str.startswith('sz')]
        if sz.empty:
            continue
        pairs = [(float(row['onset']), float(row['onset']) + float(row['duration']))
                 for _, row in sz.iterrows()
                 if float(row.get('duration', 0)) > 0]
        if pairs:
            seizures[edf_name] = pairs
    return seizures

# ── Anotações de TODOS os sujeitos SeizeIT2 ──────────────────────────────────
seizeit_ann_all = {}
for _p in PATIENTS['SeizeIT2']:
    eeg_dir = os.path.join(SEIZEIT_DIR, _p, SEIZEIT_SESSION, 'eeg')
    ann = parse_seizeit2_subject(eeg_dir) if os.path.isdir(eeg_dir) else {}
    seizeit_ann_all[_p] = ann
    print(f"✅ SeizeIT2 {_p} — {len(ann)} arquivos com crise")
    for fname, seiz in ann.items():
        for s,e in seiz:
            print(f"   {os.path.basename(fname)}  →  {s:.1f}s – {e:.1f}s  ({e-s:.1f}s)")

### 3.4 Mendeley (planilha XLSX)

In [ ]:
def _frac_day_to_sec(x):
    """A planilha do Mendeley guarda tempos como FRAÇÃO DE DIA (0..1)."""
    try:
        return float(x) * 24 * 3600
    except Exception:
        return None

def parse_mendeley_xlsx(xlsx_path, patients=None):
    """
    Lê Seizures_Information.xlsx e devolve:
        { 'p10': { 'p10_Record1.edf': [(onset_s, offset_s), ...] }, ... }
    Onset/duração estão em fração de dia; convertendo para segundos relativos
    ao início do arquivo (File onset é o início da gravação).
    """
    raw = pd.read_excel(xlsx_path, header=1)
    raw.columns = [str(c).strip() for c in raw.columns]
    # nomes esperados de colunas (robusto a variações de espaço)
    def col(part):
        for c in raw.columns:
            if part.lower() in c.lower(): return c
        return None
    c_pat   = col('Parient') or col('Patient')
    c_file  = col('File Name')
    c_fons  = col('File onset')
    c_son   = col('Seizure onset')
    c_dur   = col('Duration')
    out = {}
    cur_pat, cur_file, cur_fons = None, None, None
    for _, row in raw.iterrows():
        if c_pat and pd.notna(row.get(c_pat)):
            cur_pat = f"p{int(float(row[c_pat]))}"
        if c_file and pd.notna(row.get(c_file)):
            cur_file = str(row[c_file]).strip()
            if not cur_file.lower().endswith('.edf'):
                cur_file += '.edf'
            cur_fons = _frac_day_to_sec(row.get(c_fons)) if c_fons else 0.0
        if cur_pat is None or cur_file is None:
            continue
        if patients is not None and cur_pat not in patients:
            continue
        son = _frac_day_to_sec(row.get(c_son)) if c_son else None
        if son is None:
            continue
        # duração vem como texto 'X min Y sec' — parseia
        dsec = 0.0
        dtxt = str(row.get(c_dur, ''))
        mm = re.search(r'(\d+)\s*min', dtxt); ss = re.search(r'(\d+)\s*sec', dtxt)
        if mm: dsec += int(mm.group(1)) * 60
        if ss: dsec += int(ss.group(1))
        if dsec <= 0:
            continue
        onset_rel = max(0.0, son - (cur_fons or 0.0))
        out.setdefault(cur_pat, {}).setdefault(cur_file, []).append((onset_rel, onset_rel + dsec))
    return out

_mend_xlsx = os.path.join(MENDELEY_DIR, 'Documentation', 'Seizures_Information.xlsx')
mendeley_ann_all = {}
if os.path.exists(_mend_xlsx):
    parsed = parse_mendeley_xlsx(_mend_xlsx, PATIENTS['Mendeley'])
    for p in PATIENTS['Mendeley']:
        mendeley_ann_all[p] = parsed.get(p, {})
        print(f"✅ Mendeley {p} — {len(mendeley_ann_all[p])} arquivos com crise")
        for fn, seiz in mendeley_ann_all[p].items():
            for s,e in seiz:
                print(f"   {fn}  →  {s:.0f}s – {e:.0f}s  ({e-s:.0f}s)")
else:
    print("⚠️  Seizures_Information.xlsx não encontrado — rode o download antes.")

## 4 — Carregamento de EDF + reamostragem para fs comum

Siena é 512 Hz e Mendeley 500 Hz; os demais 256 Hz. Reamostramos **tudo para 256 Hz** para que as features espectrais sejam comparáveis e para poder juntar datasets no treino.

In [ ]:
_NON_EEG = {'ecg','emg','acc','gyr','mov','resp','ekg','eog','chest','abd',
            'eda','temp','bvp','hr','ppg'}

def load_edf(path, scale_uv=True):
    """Carrega EDF, mantém só canais EEG. Retorna (data, sfreq, ch_names, dur_s)."""
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    try:
        raw.pick('eeg')
    except Exception:
        keep = [c for c in raw.ch_names if not any(p in c.lower() for p in _NON_EEG)]
        if keep: raw.pick(keep)
    data  = raw.get_data() * (1e6 if scale_uv else 1.0)
    sfreq = float(raw.info['sfreq'])
    return data, sfreq, list(raw.ch_names), data.shape[1] / sfreq

def resample_to(data, sfreq, target=TARGET_FS):
    """Reamostra (n_ch, n_samp) para target Hz via polifase. Mantém se já igual."""
    if int(sfreq) == int(target):
        return data, float(target)
    from math import gcd
    up, down = target, int(round(sfreq))
    g = gcd(up, down); up //= g; down //= g
    out = resample_poly(data, up, down, axis=1)
    return out.astype(np.float32), float(target)

### 4.1 Inventário de canais e regiões

Confira aqui se o mapa eletrodo→região cobre os canais reais de cada dataset. Ajuste `ELECTRODE_REGION` na config se algum canal aparecer sem região.

In [ ]:
# ── Inventário de canais: ajuda a conferir o mapa eletrodo→região ────────
# Imprime os nomes de canal de 1 arquivo de cada dataset e a região inferida.
# Use isto para validar/ajustar ELECTRODE_REGION antes de extrair features.
def _first_edf(dataset):
    if dataset == 'CHBMIT':
        p = PATIENTS['CHBMIT'][0]; d = os.path.join(CHBMIT_DIR, p)
    elif dataset == 'Siena':
        p = PATIENTS['Siena'][0]; d = os.path.join(SIENA_DIR, p)
    elif dataset == 'Mendeley':
        d = os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')
    elif dataset == 'SeizeIT2':
        p = PATIENTS['SeizeIT2'][0]; d = os.path.join(SEIZEIT_DIR, p, SEIZEIT_SESSION, 'eeg')
    else:
        return None
    if not os.path.isdir(d): return None
    edfs = [f for f in os.listdir(d) if f.lower().endswith('.edf')]
    return os.path.join(d, edfs[0]) if edfs else None

for ds in ['CHBMIT','Siena','Mendeley','SeizeIT2']:
    path = _first_edf(ds)
    if not path:
        print(f"⚠️  {ds}: nenhum EDF encontrado ainda."); continue
    _, sf, ch, _ = load_edf(path)
    reg = regions_present(ch, ds)
    print(f"\n▸ {ds}  (fs={sf:.0f}Hz, {len(ch)} canais)")
    print(f"   canais: {ch[:8]}{' ...' if len(ch)>8 else ''}")
    print("   regiões:", {r: len(idx) for r, idx in reg.items() if idx})

## 5 — Pré-processamento + Rotulagem + Salvamento Nível 1 (L1)

Notch (60Hz CHB-MIT, 50Hz demais) → passa-alta 0.5Hz → passa-baixa 40Hz. Rotula cada amostra e salva o L1 por arquivo.

In [ ]:
def notch_filter(data, sfreq, freq, Q=30):
    b, a = iirnotch(freq, Q, sfreq); return filtfilt(b, a, data, axis=-1)
def highpass_filter(data, sfreq, cutoff=F_HP, order=F_ORDER):
    sos = butter(order, cutoff, btype='high', fs=sfreq, output='sos'); return sosfiltfilt(sos, data, axis=-1)
def lowpass_filter(data, sfreq, cutoff=F_LP, order=F_ORDER):
    sos = butter(order, cutoff, btype='low', fs=sfreq, output='sos'); return sosfiltfilt(sos, data, axis=-1)
def preprocess(data, sfreq, notch_freq):
    """Notch → passa-alta 0.5Hz → passa-baixa 40Hz."""
    out = notch_filter(data, sfreq, notch_freq)
    out = highpass_filter(out, sfreq)
    out = lowpass_filter(out, sfreq)
    return out

def build_label_array(n_samples, sfreq, seizure_intervals,
                      pre_sec=PRE_SEC, post_sec=POST_SEC, igap_sec=IGAP_SEC):
    """Labels amostra-a-amostra: interictal/pré/ictal/pós + unknown (margem)."""
    from scipy.ndimage import uniform_filter1d
    labels = np.full(n_samples, LBL['unknown'], dtype=np.int8)
    for (s_s, e_s) in seizure_intervals:
        i_s = int(s_s * sfreq); i_e = min(int(e_s * sfreq), n_samples)
        labels[i_s:i_e] = LBL['ictal']
        pre = labels[max(0, i_s-int(pre_sec*sfreq)):i_s];   pre[pre != LBL['ictal']] = LBL['preictal']
        pos = labels[i_e:min(n_samples, i_e+int(post_sec*sfreq))]; pos[pos != LBL['ictal']] = LBL['postictal']
    ev = (labels != LBL['unknown']).astype(np.float32)
    gap = int(igap_sec * sfreq)
    near = (uniform_filter1d(ev, size=gap*2+1, mode='constant') > 0) if (gap>0 and ev.any()) else ev.astype(bool)
    labels[(~near) & (labels == LBL['unknown'])] = LBL['interictal']
    return labels

def save_level1(data_filt, labels, sfreq, ch_names, dataset, patient, fname, out_dir=L1_DIR):
    key  = f'{dataset}__{patient}__{os.path.splitext(os.path.basename(fname))[0]}'
    npz  = os.path.join(out_dir, key + '_L1.npz')
    if not os.path.exists(npz):
        np.savez_compressed(npz, data=data_filt.astype(np.float32), labels=labels,
                            sfreq=np.float32(sfreq), ch_names=np.array(ch_names),
                            dataset=dataset)
        print(f"💾 L1: {os.path.basename(npz)}")
    else:
        print(f"⏭️  Já existe: {os.path.basename(npz)}")
    return npz

def edf_root(dataset, patient):
    if dataset=='CHBMIT':   return os.path.join(CHBMIT_DIR, patient)
    if dataset=='Siena':    return os.path.join(SIENA_DIR, patient)
    if dataset=='Mendeley': return os.path.join(MENDELEY_DIR, 'Raw_EDF_Files')
    if dataset=='SeizeIT2': return os.path.join(SEIZEIT_DIR, patient, SEIZEIT_SESSION, 'eeg')

l1_index = {}
def run_l1(dataset, patient, ann_dict):
    for fname, seiz in ann_dict.items():
        path = os.path.join(edf_root(dataset, patient), fname)
        if not os.path.exists(path):
            print(f"  ⚠️  {fname} não encontrado"); continue
        data, sfreq, ch, _ = load_edf(path)
        data, sfreq = resample_to(data, sfreq, TARGET_FS)   # fs comum
        data_f = preprocess(data, sfreq, NOTCH[dataset])
        labels = build_label_array(data_f.shape[1], sfreq, seiz)
        npz = save_level1(data_f, labels, sfreq, ch, dataset, patient, fname)
        l1_index[f'{dataset}__{patient}__{os.path.splitext(os.path.basename(fname))[0]}'] = npz

ANN = {'CHBMIT': chbmit_ann_all, 'Siena': siena_ann_all,
       'Mendeley': mendeley_ann_all, 'SeizeIT2': seizeit_ann_all}
for ds, ann_all in ANN.items():
    for pat, ann in ann_all.items():
        if ann: run_l1(ds, pat, ann)
print(f"\n📦 Total L1: {len(l1_index)}")
json.dump({k: os.path.basename(v) for k,v in l1_index.items()},
          open(os.path.join(L1_DIR, '_index.json'),'w'), indent=2)

---
✅ **Fim do Notebook 1.** O L1 está em `data/level1_signals/`. Prossiga para o **Notebook 2 — Janelas, Níveis de Canal e Features**.